# Asymmetric Multilingual Acquisition: Typology-Aware Adaptive Mixing for BabyLMs

- Get HF_TOKEN with `read-access`

In [ ]:
# Step 1 — Parameters (edit these)
# --------------------------------

# --- Repository ---------------------------------------------------------
REPO_URL    = "https://github.com/Amos-Luna/Asymmetric-Multilingual-Acquisition_TAAM.git"
REPO_BRANCH = "main"
REPO_DIR    = "/content/taam"        # where the repo is cloned

# --- Persistence (Google Drive) ----------------------------------------
# We persist heavy artifacts under MyDrive/Researchs/BabyLM/ so:
#   - data/ + tokenizer/ are downloaded ONCE and reused by every run.
#   - runs/{date}_{cond}_seed{S}/ keeps every experiment isolated.
# If you want a different parent folder, change DRIVE_ROOT below.
USE_DRIVE   = True                   # False = ephemeral disk only (lost on disconnect)
DRIVE_ROOT  = "/content/drive/MyDrive/Researchs/BabyLM"

# --- What to run -------------------------------------------------------
MODE        = "single"               # "single" = one condition; "matrix" = all 33 runs
# Available SINGLE_COND values (one config file per id, see configs/):
#   Headline    : TAAM       (typology prior + EXP3 + v1 reward, pre-registered)
#                 TAAM_v2    (TAAM with v2 cross-lingual-deficit reward; the
#                             reward fix for the v1 collapse failure mode)
#   Ablations   : B0, B1, B2, O1, O2, P1, P2
#   Controls    : Pstar      (static mixture frozen at TAAM-v2's time-averaged
#                             converged allocation ~.16/.26/.58; the decisive
#                             "is it just more Chinese, or does timing matter?"
#                             control. Run with the SAME seeds as TAAM_v2.)
#   Monolingual : M_EN, M_NL, M_ZH
SINGLE_COND = "TAAM"                 # used when MODE == "single"
SINGLE_SEED = 42                     # used when MODE == "single"

# --- Training schedule -------------------------------------------------
TOTAL_STEPS = 20_000                 # CFP-compliant default (~6.5 epochs over 100M)
EVAL_EVERY  = 200                    # online reward update cadence

# --- Optional: enable WandB --------------------------------------------
USE_WANDB   = False                  # if True, also add a "WANDB_API_KEY" secret

print(
    "Configuration:\n"
    f"  repo       : {REPO_URL} @ {REPO_BRANCH}\n"
    f"  clone dir  : {REPO_DIR}\n"
    f"  Drive      : {'on (' + DRIVE_ROOT + ')' if USE_DRIVE else 'off'}\n"
    f"  mode       : {MODE}"
    + (f"  (condition={SINGLE_COND}, seed={SINGLE_SEED})" if MODE == 'single' else "")
    + "\n"
    f"  steps/run  : {TOTAL_STEPS:,}\n"
    f"  WandB      : {'on' if USE_WANDB else 'off'}"
)

In [ ]:
# Step 2 — Clone or pull the repo
# --------------------------------

import os
import subprocess
from pathlib import Path

def run(cmd: str, cwd: str | None = None, check: bool = True) -> str:
    """Run a shell command, stream output, return combined stdout/stderr."""
    print(f"$ {cmd}")
    res = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if res.stdout.strip():
        print(res.stdout)
    if res.stderr.strip():
        print(res.stderr)
    if check and res.returncode != 0:
        raise RuntimeError(f"command failed (exit {res.returncode}): {cmd}")
    return res.stdout

repo_path = Path(REPO_DIR)
if repo_path.exists() and (repo_path / ".git").exists():
    print(f"Repo exists at {repo_path}; pulling latest from origin/{REPO_BRANCH} ...")
    run("git fetch --all --tags --prune", cwd=str(repo_path))
    run(f"git checkout {REPO_BRANCH}", cwd=str(repo_path), check=False)
    run(f"git pull --ff-only origin {REPO_BRANCH}", cwd=str(repo_path))
else:
    if repo_path.exists():
        # Directory exists but is not a git repo — refuse to wipe blindly.
        raise RuntimeError(
            f"{repo_path} exists but is not a git repo. "
            "Remove it manually or change REPO_DIR."
        )
    print(f"Cloning {REPO_URL} → {repo_path} ...")
    run(f"git clone --branch {REPO_BRANCH} {REPO_URL} {repo_path}")

os.chdir(repo_path)
print(f"\nWorking directory: {Path.cwd()}")
print(f"HEAD: {run('git rev-parse --short HEAD', cwd=str(repo_path)).strip()}  "
      f"({run('git log -1 --format=%s', cwd=str(repo_path)).strip()})")

In [ ]:
# Step 3 - Auth + (optional) Drive persistence
# --------------------------------------------

import os
from pathlib import Path

# ---- HuggingFace token from Colab Secrets ----
try:
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF_TOKEN: loaded from Colab Secrets.")
    except userdata.SecretNotFoundError:
        raise RuntimeError(
            "Colab Secret 'HF_TOKEN' is not set.\n"
            "  1. Open the key icon in the left sidebar.\n"
            "  2. Add a secret named HF_TOKEN with your hf_... value.\n"
            "  3. Toggle 'Notebook access' to ON.\n"
            "Then re-run this cell."
        )
    if USE_WANDB:
        try:
            os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
            print("WANDB_API_KEY: loaded.")
        except userdata.SecretNotFoundError:
            print("WandB skipped: no WANDB_API_KEY secret. Set USE_WANDB=False or add the secret.")
except ImportError:
    # Outside Colab — fall back to env or .env.
    if not os.environ.get("HF_TOKEN") and Path(".env").exists():
        for line in Path(".env").read_text().splitlines():
            if line.startswith("HF_TOKEN="):
                os.environ["HF_TOKEN"] = line.split("=", 1)[1].strip().strip('"').strip("'")
                print("HF_TOKEN: loaded from .env (not in Colab).")
                break

# ---- Drive mount + symlinks ----
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except ImportError:
        print("Not running in Colab; skipping Drive mount.")
        USE_DRIVE = False  # type: ignore[assignment]

    if USE_DRIVE:
        drive_root = Path(DRIVE_ROOT)
        drive_root.mkdir(parents=True, exist_ok=True)
        # Heavy directories that should persist across sessions:
        for sub in ("data/hf_cache", "data/tokens", "tokenizer", "runs", "wandb"):
            (drive_root / sub).mkdir(parents=True, exist_ok=True)
            dst = Path(REPO_DIR) / sub
            dst.parent.mkdir(parents=True, exist_ok=True)
            # If dst exists and isn't a symlink, move its contents to Drive and replace.
            if dst.exists() and not dst.is_symlink():
                import shutil
                print(f"  migrating existing {dst} -> {drive_root / sub}")
                for item in dst.iterdir():
                    target = drive_root / sub / item.name
                    if not target.exists():
                        shutil.move(str(item), str(target))
                shutil.rmtree(dst)
            if dst.is_symlink():
                dst.unlink()
            dst.symlink_to(drive_root / sub, target_is_directory=True)
        print(f"Drive symlinks created under {drive_root}")
else:
    print("Drive disabled (USE_DRIVE=False). Data and runs will be lost on disconnect.")

In [ ]:
# Step 4 - Bootstrap (install + verify + data pipeline + GPU smoke)
# skipped automatically if Drive already has the artifacts
# --------------------------------------------------------------

bootstrap_args = []
if not USE_DRIVE:
    bootstrap_args.append("--no-drive")
else:
    bootstrap_args += ["--drive-root", DRIVE_ROOT]

# Stream the bootstrap output live so you see progress (not just at the end).
import subprocess, sys
proc = subprocess.Popen(
    ["bash", "scripts/colab_bootstrap.sh", *bootstrap_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"colab_bootstrap.sh exited with code {rc}")

In [ ]:
# Step 5 - Train
# - **`MODE = "single"`** runs ONE condition × seed. Use this for the first end-to-end smoke (TAAM seed 42 is the headline), or for iterating on a single experiment.
# - **`MODE = "matrix"`** runs the full 11 × 3 = 33-run matrix via `scripts/run_matrix.py`. Resumable: if you disconnect midway, re-running the notebook picks up where it left off (per-run `DONE` markers).

#Wall-clock estimates (from `gpu_smoke_test.py` are printed above):
#- H100: ~25-40 min per 20k-step run; full matrix ~14-22 h.
#- A100: ~50-70 min per run; full matrix ~28-38 h (consider splitting across sessions).
#- T4 / Colab Free: don't. Use bigger GPU.

import subprocess

if MODE == "single":
    # NB: we deliberately omit --output-dir so train.py auto-names the run
    # directory as runs/{YYYY-MM-DD}_{COND}_seed{S}/. That naming convention
    # works with the resume logic in scripts/run_matrix.py.
    cmd = [
        "python", "scripts/train.py",
        "--condition", SINGLE_COND,
        "--seed", str(SINGLE_SEED),
        "--token-data", "data/tokens",
        "--tokenizer", "tokenizer/spm_32k_en_nl_zh.model",
        "--total-steps", str(TOTAL_STEPS),
        "--eval-every", str(EVAL_EVERY),
    ]
elif MODE == "matrix":
    cmd = [
        "python", "scripts/run_matrix.py",
        "--total-steps", str(TOTAL_STEPS),
        "--eval-every", str(EVAL_EVERY),
        "--output-dir", "runs",
        "--retries", "2",
    ]
else:
    raise ValueError(f"unknown MODE: {MODE!r}; expected 'single' or 'matrix'")

print("Launching:")
print("  " + " ".join(cmd) + "\n")

proc = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"training exited with code {rc} (check the log above)")
print("\n✓ Training cell completed.")

In [ ]:
# Step 6 — Paper-quality Figure 1 (auto from the artifacts we just wrote)
# ----------------------------------------------------------------------
# Reads pi_history.csv + log.jsonl + config.yaml + summary.json from the
# run dir we just produced, then renders the annotated two-panel diagnostic:
#
#   Top panel    — pi_l(t) with phase bands (cold start / discovery /
#                  plateau / collapse), the typological prior pi_0 (dashed),
#                  the uniform 1/K reference (dotted), and the asymptotic
#                  EXP3 structural floor (solid red, computed from gamma +
#                  min_pi in the run's own config).
#   Bottom panel — per-language val loss with the same phase bands.
#
# WHERE THE FIGURE IS SAVED
# -------------------------
# Primary copy  → INSIDE the run dir (e.g. runs/2026-05-14_TAAM_seed42/figure1.png).
#                 Because `runs/` is symlinked to your Drive by the bootstrap,
#                 this lives at $DRIVE_ROOT/runs/<run-name>/figure1.png and
#                 SURVIVES the runtime disconnect.
# Backup copy   → analyses/figures/fig1_<cond>_seed<seed>.png in the repo
#                 working dir. This is EPHEMERAL (lost on disconnect) but
#                 handy for in-session sharing / paper compilation.
# Both PNG and PDF (vector) are written in each location.

import importlib
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt

# Make sure REPO_DIR is on sys.path (already chdir'd in Step 2; defensive).
repo_root = Path(REPO_DIR).resolve() if "REPO_DIR" in globals() else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Re-import so an in-session edit to the figure script picks up.
import analyses.figures.figure1_taam_diagnostic as fig1mod
importlib.reload(fig1mod)

# Locate the run dir we just produced (most recent match to cond × seed).
target_cond = SINGLE_COND if MODE == "single" else "TAAM"
target_seed = SINGLE_SEED if MODE == "single" else 42
candidates = sorted(
    p for p in (repo_root / "runs").glob(f"*_{target_cond}_seed{target_seed}") if p.is_dir()
)
if not candidates:
    raise FileNotFoundError(
        f"No run dir found under runs/*_{target_cond}_seed{target_seed}. "
        f"Did Step 5 finish? Existing dirs: "
        f"{sorted(p.name for p in (repo_root / 'runs').iterdir() if p.is_dir())}"
    )
run_dir = candidates[-1]
print(f"Run dir       : {run_dir}")
print(f"  contents    : {sorted(p.name for p in run_dir.iterdir())}")

# Primary output: inside the run dir (this is on Drive via the runs/ symlink).
primary_png = run_dir / "figure1.png"

run = fig1mod.load_run(run_dir)
fig = fig1mod.make_figure(
    run,
    output=primary_png,
    show=True,  # do NOT close the figure; let Jupyter render it inline.
)
plt.show()

# Backup copy in the repo's analyses/ folder (ephemeral on Colab; useful
# during the session for quick drag-into-paper or notebook downloads).
backup_dir = repo_root / "analyses" / "figures"
backup_dir.mkdir(parents=True, exist_ok=True)
backup_png = backup_dir / f"fig1_{target_cond}_seed{target_seed}.png"
backup_pdf = backup_dir / f"fig1_{target_cond}_seed{target_seed}.pdf"
shutil.copy2(primary_png, backup_png)
shutil.copy2(primary_png.with_suffix(".pdf"), backup_pdf)


# ── Print where everything actually lives ──────────────────────────────
def _drive_url(local_path: Path) -> str:
    """Translate a /content/.../runs/... path into its real Drive location."""
    # The repo's runs/ symlink resolves to $DRIVE_ROOT/runs/.
    resolved = local_path.resolve()
    if "/drive/MyDrive/" in str(resolved):
        rel = str(resolved).split("/drive/MyDrive/", 1)[1]
        return f"Mi unidad / {rel.replace('/', ' / ')}"
    return "(ephemeral disk — not on Drive)"


print()
print("─" * 72)
print("Figure 1 was saved to:")
print()
print("  ● Primary (persisted on Google Drive):")
print(f"      Colab path : {primary_png}")
print(f"      Drive path : {primary_png.resolve()}")
print(f"      Drive UI   : {_drive_url(primary_png)}")
print(f"      (+ vector  : {primary_png.with_suffix('.pdf').name})")
print()
print("  ● Backup (ephemeral, lost on disconnect):")
print(f"      Repo path  : {backup_png}")
print(f"      (+ vector  : {backup_pdf.name})")
print("─" * 72)

# ── Headline numbers under the figure ─────────────────────────────────
K = len(run.languages)
floor = fig1mod.structural_floor(run.gamma, run.min_pi, K)
final_pi = dict(zip(run.languages, run.pi[-1].tolist()))

print()
print(f"Condition     : {(run.summary or {}).get('condition_id', target_cond)}")
print(f"Reward        : {run.reward_type}")
print(f"Total steps   : {(run.summary or {}).get('total_steps', int(run.pi_steps[-1]))}")
print(f"gamma         : {run.gamma:.3f}")
print(f"min_pi        : {run.min_pi:.3f}")
print(f"Structural floor (asymptotic min π if soft→0) : {floor:.4f}")
print()
print(f"Final π  : " + "  ".join(f"{l}={v:.3f}" for l, v in final_pi.items()))
if run.val_loss is not None:
    final_val = {l: float(v[-1]) for l, v in run.val_loss.items()}
    print(f"Final val: " + "  ".join(f"{l}={v:.3f}" for l, v in final_val.items()))
    best_lang = min(final_val, key=final_val.get)
    print(
        f"\nDeficit vs. easiest language ({best_lang}, val={final_val[best_lang]:.3f}):"
    )
    for l, v in final_val.items():
        print(f"  {l}: +{v - final_val[best_lang]:.3f} nats")

# Flag the failure mode explicitly when it triggers.
zho_final = final_pi.get("zho")
if zho_final is not None and zho_final <= floor + 0.01:
    print(
        "\n[WARNING] π(zho) is pinned to the EXP3 structural floor "
        f"(π_zho={zho_final:.4f} vs floor={floor:.4f}). "
        "This is the v1-reward failure mode (see taam/reward.py); consider "
        "rerunning with the v2 reward via condition='TAAM_v2'."
    )